<a href="https://colab.research.google.com/github/MichelleIhetu/BMEN-499_AlphaFold/blob/main/BMEN_499.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request
import json
import os

In [ ]:
PROTEINS = [
    {"name": "Lysozyme",           "uniprot_id": "P00720", "function": "Antimicrobial enzyme; cleaves bacterial cell walls"},
    {"name": "Insulin",            "uniprot_id": "P01308", "function": "Hormone; regulates blood glucose levels"},
    {"name": "Myoglobin",          "uniprot_id": "P02144", "function": "Oxygen storage in muscle tissue"},
    {"name": "Ubiquitin",          "uniprot_id": "P0CG47", "function": "Protein degradation signal; tags proteins for proteasomal destruction"},
    {"name": "Calmodulin",         "uniprot_id": "P0DP23", "function": "Calcium-binding messenger protein; regulates many enzymes"},
    {"name": "Thioredoxin",        "uniprot_id": "P10599", "function": "Antioxidant; reduces oxidized proteins via cysteine residues"},
    {"name": "Cytochrome C",       "uniprot_id": "P99999", "function": "Electron carrier in the mitochondrial respiratory chain"},
    {"name": "Green Fluorescent Protein (GFP)", "uniprot_id": "P42212", "function": "Fluorescent reporter; emits green light upon UV excitation"},
    {"name": "Barnase",            "uniprot_id": "P00648", "function": "Ribonuclease; degrades RNA in bacterial cells"},
    {"name": "Adenylate Kinase",   "uniprot_id": "P69441", "function": "Phosphotransferase; maintains cellular energy balance (ATP/ADP/AMP)"},
]
print(PROTEINS)

[{'name': 'Lysozyme', 'uniprot_id': 'P00720', 'function': 'Antimicrobial enzyme; cleaves bacterial cell walls'}, {'name': 'Insulin', 'uniprot_id': 'P01308', 'function': 'Hormone; regulates blood glucose levels'}, {'name': 'Myoglobin', 'uniprot_id': 'P02144', 'function': 'Oxygen storage in muscle tissue'}, {'name': 'Ubiquitin', 'uniprot_id': 'P0CG47', 'function': 'Protein degradation signal; tags proteins for proteasomal destruction'}, {'name': 'Calmodulin', 'uniprot_id': 'P0DP23', 'function': 'Calcium-binding messenger protein; regulates many enzymes'}, {'name': 'Thioredoxin', 'uniprot_id': 'P10599', 'function': 'Antioxidant; reduces oxidized proteins via cysteine residues'}, {'name': 'Cytochrome C', 'uniprot_id': 'P99999', 'function': 'Electron carrier in the mitochondrial respiratory chain'}, {'name': 'Green Fluorescent Protein (GFP)', 'uniprot_id': 'P42212', 'function': 'Fluorescent reporter; emits green light upon UV excitation'}, {'name': 'Barnase', 'uniprot_id': 'P00648', 'functi

In [ ]:
ALPHAFOLD_API = "https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
UNIPROT_API   = "https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
OUTPUT_DIR    = "alphafold_results"

In [ ]:
def fetch_json(url: str) -> dict | None:
    """Fetch JSON from a URL; return None on failure."""
    try:
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            return json.loads(resp.read().decode())
    except Exception as exc:
        print(f"[ERROR] {exc}")
        return None

In [ ]:
def fetch_alphafold(uniprot_id: str) -> dict | None:
    """Query the AlphaFold DB API for a given UniProt accession."""
    url  = ALPHAFOLD_API.format(uniprot_id=uniprot_id)
    data = fetch_json(url)
    if data and isinstance(data, list) and len(data) > 0:
        return data[0]
    return None

In [ ]:
def fetch_uniprot_length(uniprot_id: str) -> int | None:
    """Return the sequence length from UniProt."""
    url  = UNIPROT_API.format(uniprot_id=uniprot_id)
    data = fetch_json(url)
    if data:
        return data.get("sequence", {}).get("length")
    return None

In [ ]:
def download_pdb(pdb_url: str, dest_path: str) -> bool:
    """Download a PDB file to dest_path."""
    try:
        urllib.request.urlretrieve(pdb_url, dest_path)
        return True
    except Exception as exc:
        print(f"    [ERROR] Could not download PDB: {exc}")
        return False


In [ ]:
import os
def main():
  os.makedirs(OUTPUT_DIR, exist_ok=True)
  results = []

  print("=" * 65)
  print("AlphaFold Protein Fetcher — 10 Proteins with Known Functions")
  print("=" * 65)

  for i, protein in enumerate(PROTEINS, start=1):
        name = protein["name"]
        uid = protein["uniprot_id"]
        function = protein["function"]

        print(f"\n[{i:02d}] {name}  (UniProt: {uid})")
        print(f"      Function : {function}")

        af_data = fetch_alphafold(uid)
        if af_data:
            model_url       = af_data.get("pdbUrl", "N/A")
            confidence_url  = af_data.get("cifUrl", "N/A")
            model_version   = af_data.get("latestVersion", "N/A")
            organism        = af_data.get("organismScientificName", "N/A")
            print(f"      Organism : {organism}")
            print(f"      AF Model : v{model_version}")
            print(f"      PDB URL  : {model_url}")

            # Download PDB structure
            if model_url != "N/A":
                pdb_file = os.path.join(OUTPUT_DIR, f"{uid}.pdb")
                ok = download_pdb(model_url, pdb_file)
                print(f"      PDB File : {'saved → ' + pdb_file if ok else 'download failed'}")
        else:
            print("      AlphaFold: No entry found")
            model_url = organism = model_version = "N/A"

        length = fetch_uniprot_length(uid)
        print(f"      Length   : {length} aa" if length else "      Length   : N/A")

        results.append({
            "rank":         i,
            "name":         name,
            "uniprot_id":   uid,
            "function":     function,
            "organism":     organism if af_data else "N/A",
            "sequence_len": length,
            "af_version":   model_version if af_data else "N/A",
            "pdb_url":      model_url if af_data else "N/A"
        })
  return results

In [ ]:
import urllib.request
import json


In [ ]:
import urllib.request
import json


In [ ]:
import os
def main():
  os.makedirs(OUTPUT_DIR, exist_ok=True)
  results = []

  print("=" * 65)
  print("AlphaFold Protein Fetcher — 10 Proteins with Known Functions")
  print("=" * 65)

  for i, protein in enumerate(PROTEINS, start=1):
        name = protein["name"]
        uid = protein["uniprot_id"]
        function = protein["function"]

        print(f"\n[{i:02d}] {name}  (UniProt: {uid})")
        print(f"      Function : {function}")

        af_data = fetch_alphafold(uid)
        if af_data:
            model_url       = af_data.get("pdbUrl", "N/A")
            confidence_url  = af_data.get("cifUrl", "N/A")
            model_version   = af_data.get("latestVersion", "N/A")
            organism        = af_data.get("organismScientificName", "N/A")
            print(f"      Organism : {organism}")
            print(f"      AF Model : v{model_version}")
            print(f"      PDB URL  : {model_url}")

            # Download PDB structure
            if model_url != "N/A":
                pdb_file = os.path.join(OUTPUT_DIR, f"{uid}.pdb")
                ok = download_pdb(model_url, pdb_file)
                print(f"      PDB File : {'saved → ' + pdb_file if ok else 'download failed'}")
        else:
            print("      AlphaFold: No entry found")
            model_url = organism = model_version = "N/A"

        length = fetch_uniprot_length(uid)
        print(f"      Length   : {length} aa" if length else "      Length   : N/A")

        results.append({
            "rank":         i,
            "name":         name,
            "uniprot_id":   uid,
            "function":     function,
            "organism":     organism if af_data else "N/A",
            "sequence_len": length,
            "af_version":   model_version if af_data else "N/A",
            "pdb_url":      model_url if af_data else "N/A"
        })
  return results

In [ ]:
if __name__ == "__main__":
    final_results = main()
    summary_path = os.path.join(OUTPUT_DIR, "summary.json")
    with open(summary_path, "w") as fh:
        json.dump(final_results, fh, indent=2)

    print("\n" + "=" * 65)
    print(f"  Done. Summary saved to: {summary_path}")
    print(f"  PDB files (if downloaded) saved to: {OUTPUT_DIR}/")
    print("=" * 65)

In [ ]:
if __name__ == "__main__":
    final_results = main()
    summary_path = os.path.join(OUTPUT_DIR, "summary.json")
    with open(summary_path, "w") as fh:
        json.dump(final_results, fh, indent=2)

    print("\n" + "=" * 65)
    print(f"  Done. Summary saved to: {summary_path}")
    print(f"  PDB files (if downloaded) saved to: {OUTPUT_DIR}/")
    print("=" * 65)

NameError: name 'os' is not defined